In [13]:
import sys
sys.path.append("../../")

from ingestion.product.product_indexer import load_documents, create_chunks, embeddings, persist
from retrieval.base_retriever import get_collections
from src.core.utils import get_required_env

In [14]:
documents = load_documents("../../data/raw/product.csv", "../../data/processed/product.csv")
chunks = create_chunks(documents)
nodes = embeddings(chunks)
persist(nodes)

/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
collection = get_collections(collection_name=get_required_env("PRODUCT_COLLECTION_NAME"))

In [21]:
response = collection.query.fetch_objects(limit=10000)

In [22]:
import pandas as pd

rows = []
for obj in response.objects:
    rows.append({
        "uuid": str(obj.uuid),
        "product_id": obj.properties.get("product_id"),
        "brand": obj.properties.get("brand"),
        "color": obj.properties.get("color"),
        "category": obj.properties.get("category"),
        "price": obj.properties.get("price"),
        "avg_rating": obj.properties.get("avg_rating"),
        "text": obj.properties.get("text"),
    })

df = pd.DataFrame(rows)
df.head(10)

,uuid,product_id,brand,color,category,price,avg_rating,text
0,0004e6f6-2d6c-58f2-ac18-b9b530009807,17812474,WISSTLER,Rust,None,1199.0,4.0,Product: WISSTLER Women Rust Printed Longline ...
1,001de66b-52ba-513f-b8dd-401724f42144,13617668,Shaily,Black,None,1789.0,None,Product: Shaily Black & White Printed Dupatta\...
2,00231f26-906f-541f-ac1e-a616a160889a,18458340,BoStreet,Blue,Denim shorts,1999.0,None,Product: BoStreet Women Blue Slim Fit High-Ris...
3,00249519-0ca0-5f7a-818f-36efa4ed0c9d,7592398,SOUNDARYA,Lime Green,None,999.0,3.7,Product: SOUNDARYA Lime Green & Pink Dyed Pure...
4,0026d62a-991c-5192-ac08-a27045480772,18067174,BUY NEW TREND,Brown,None,1399.0,None,Product: BUY NEW TREND Women Brown High-Low Sh...
5,002729f2-51bf-5288-bf6e-55509b3fd9ce,18996592,Popwings,Black,None,1299.0,None,Product: Popwings Women Black Rayon Regular Fi...
6,0032af0e-d21c-5622-b876-c5c8e5078e3f,13938274,SHUBHKALA,Beige,None,12000.0,None,Product: SHUBHKALA Beige & Golden Sequinned Se...
7,003695e8-4907-55b8-bbf1-401adfe098e1,13957492,Clora Creation,White,Flared,1799.0,None,Product: Clora Creation Women White & Red Prin...
8,0039bf95-a04e-5f48-80b0-c8bc8891c9b7,10898748,MOKSHA DESIGNS,Maroon,None,47999.0,None,Product: MOKSHA DESIGNS Maroon & Pink Embroide...
9,003a8aa4-82f0-5b58-81b8-aaf009253ebb,14810232,TARAMA,Blue,None,2999.0,3.7777777777777777,Product: TARAMA Women Blue Straight Fit High-R...


In [24]:
import json
from pathlib import Path

output_path = Path("data/processed/collection.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

rows = [
    {
        "uuid": str(obj.uuid),
        **obj.properties,
    }
    for obj in response.objects
]

with output_path.open("w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)